# Reducción Dimensional con Criterio Spearman  

# Prompt  

Dame un script de Python para realizar la reducción dimensional de los atributos meteorológicos con y sin rezago, del archivo .xlsx que te adjunto, donde la variable objetivo es `casos_dengue` y teniendo en cuanta que queremos los atributos meteorológicos mejor correlacionados con esta variable objetivo pero menos correlacionados entre sí. Como los valores de correlación son bajos, pon los umbrales de correlación de acuerdo a los dos mejores percentiles.  

Aquí tienes un script de Python estructurado para lograr exactamente lo que buscas.

Dado que mencionas que las correlaciones generales son bajas, el script calcula dinámicamente los **percentiles más altos** para tomar decisiones automatizadas:

1. **Filtro de Relevancia:** Selecciona las variables meteorológicas que tengan una correlación con `casos_dengue` por encima del **percentil 90** (el top 10% de las mejores correlaciones).
2. **Filtro de Redundancia:** Entre las variables seleccionadas, evalúa su correlación interna (multicolinealidad). Si dos variables están muy correlacionadas entre sí (superando el **percentil 25** de las correlaciones del set filtrado), elimina la que tenga menor impacto sobre los casos de dengue.

### Script de Python

```python


In [22]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset (ajusta la ruta según corresponda)
# Nota: Como especificaste archivo .xlsx, usamos read_excel. Si usas el .csv, cambia a read_csv.
df = pd.read_excel(r"C:\Users\marco\Documentos\investigacion\arima\07_seleccion_entrenar_modelo\1_reduccion_dimensional\2_spearman_2021_2025\2_data\1_raw\1_meteo_epi_rezagos.xlsx")

# 2. Definir variables
target = "casos_dengue"

# Excluir variables de control/tiempo para quedarnos solo con las meteorológicas (y sus lags)
exclude_cols = ["fecha", "año", "semana_epi", "soi", "sst", target]
features = [col for col in df.columns if col not in exclude_cols]

# Asegurar que los datos sean numéricos y no tengan nulos para el cálculo de correlación
df_numeric = df[features + [target]].dropna()

# 3. Calcular la matriz de correlación completa
corr_matrix = df_numeric.corr(method="pearson")

# --- PASO 1: Filtrar por relevancia con la variable objetivo ---
# Obtener la correlación absoluta con 'casos_dengue'
target_corr = corr_matrix[target].abs().drop(target)

# Definir el umbral usando el percentil alto (ej. Percentil 90 = Top 10% mejores predictores)
threshold_target = target_corr.quantile(0.60)
selected_features = target_corr[target_corr >= threshold_target].index.tolist()

print(f"--- PASO 1: SELECCIÓN POR RELEVANCIA ---")
print(f"Umbral de correlación con el objetivo (Percentil 90): {threshold_target:.4f}")
print(f"Atributos seleccionados ({len(selected_features)}): {selected_features}\n")

# --- PASO 2: Filtrar por redundancia (menos correlacionados entre sí) ---
# Matriz de correlación solo entre las variables preseleccionadas
sub_corr_matrix = df_numeric[selected_features].corr().abs()

# Extraer los valores de la diagonal superior (para no duplicar parejas ni evaluar la diagonal de 1s)
upper_tri = sub_corr_matrix.where(np.triu(np.ones(sub_corr_matrix.shape), k=1).astype(bool))
all_corrs = upper_tri.stack().values

# Definir el umbral de redundancia usando un percentil bajo de las correlaciones existentes
# Como las correlaciones son bajas, usamos el percentil 25 de sus intercorrelaciones para ser estrictos
threshold_redundancy = np.nanpercentile(all_corrs, 90)

print(f"--- PASO 2: FILTRADO POR REDUNDANCIA ---")
print(f"Umbral máximo permitido entre atributos (Percentil 25): {threshold_redundancy:.4f}")

# Algoritmo para eliminar características redundantes
features_to_remove = set()
for col in sub_corr_matrix.columns:
    if col in features_to_remove:
        continue
    # Buscar variables altamente correlacionadas con la variable actual
    redundant_cols = sub_corr_matrix.index[(sub_corr_matrix[col] > threshold_redundancy) & (sub_corr_matrix.index != col)].tolist()
    
    for r_col in redundant_cols:
        if r_col not in features_to_remove:
            # Comparar cuál tiene peor correlación con el dengue y eliminarla
            if target_corr[col] >= target_corr[r_col]:
                features_to_remove.add(r_col)
            else:
                features_to_remove.add(col)
                break  # Si la variable actual es eliminada, dejamos de evaluarla

final_features = [f for f in selected_features if f not in features_to_remove]

print(f"Atributos eliminados por alta redundancia: {list(features_to_remove)}")
print(f"\n--- RESULTADO FINAL ---")
print(f"Atributos óptimos finales ({len(final_features)}): {final_features}")

# 4. Crear el nuevo dataset con la reducción dimensional realizada
df_reducido = df[["fecha", "año", "semana_epi", target] + final_features]
df_reducido.to_csv(r"C:\Users\marco\Documentos\investigacion\arima\07_seleccion_entrenar_modelo\1_reduccion_dimensional\2_spearman_2021_2025\2_data\2_processed\2_meteo_epi_rezagos_spearman.csv", index=False)


--- PASO 1: SELECCIÓN POR RELEVANCIA ---
Umbral de correlación con el objetivo (Percentil 90): 0.2442
Atributos seleccionados (44): ['hum_esp', 'hum_rel', 'temp_lag_3', 'temp_lag_4', 'temp_lag_5', 'temp_lag_6', 'temp_lag_7', 'temp_lag_8', 'temp_max_lag_2', 'temp_max_lag_3', 'temp_max_lag_4', 'temp_max_lag_5', 'temp_max_lag_6', 'temp_max_lag_7', 'temp_max_lag_8', 'hum_esp_lag_1', 'hum_esp_lag_2', 'hum_esp_lag_3', 'hum_esp_lag_4', 'hum_esp_lag_5', 'hum_esp_lag_6', 'hum_esp_lag_7', 'hum_esp_lag_8', 'hum_rel_lag_1', 'hum_rel_lag_2', 'hum_rel_lag_3', 'hum_rel_lag_4', 'hum_rel_lag_5', 'hum_rel_lag_6', 'hum_rel_lag_7', 'hum_rel_lag_8', 'dias_lluvia_lag_4', 'dias_lluvia_lag_5', 'dias_lluvia_lag_6', 'dias_lluvia_lag_7', 'dias_lluvia_lag_8', 'uv_lag_1', 'uv_lag_2', 'uv_lag_3', 'uv_lag_4', 'uv_lag_5', 'uv_lag_6', 'uv_lag_7', 'uv_lag_8']

--- PASO 2: FILTRADO POR REDUNDANCIA ---
Umbral máximo permitido entre atributos (Percentil 25): 0.8418
Atributos eliminados por alta redundancia: ['hum_esp_lag_



# ¿Cómo funciona la lógica de percentiles instalada?

* **`target_corr.quantile(0.90)`**: Encuentra el valor de correlación que supera al **90%** de las demás variables con respecto al dengue. Esto garantiza que solo te quedas con la "crema y nata" de tus atributos climáticos (con o sin rezago).
* **`np.percentile(all_corrs, 25)`**: Al analizar cómo se correlacionan estas variables seleccionadas entre sí, calcula el percentil 25. Al usar este valor como límite máximo permitido, el script se vuelve muy estricto: si dos variables se correlacionan un poco más de lo normal (rebasando ese primer cuartil), el script descarta inmediatamente la que tenga menor impacto sobre los `casos_dengue`.

In [11]:
threshold_redundancy = np.nanpercentile(all_corrs, 25)
threshold_redundancy

np.float64(0.7596565501170511)

In [10]:
all_corrs

array([       nan, 0.89693892, 0.83935043, 0.81434284, 0.77722272,
       0.72114813, 0.66071998, 0.71761334, 0.67827243, 0.6212828 ,
       0.54476459,        nan,        nan, 0.89879159, 0.84182575,
       0.81604605, 0.77936764, 0.72180111, 0.75908106, 0.72037328,
       0.68336004, 0.62519379,        nan,        nan,        nan,
       0.90016634, 0.84224912, 0.81701615, 0.77759611, 0.81528122,
       0.76023204, 0.72462857, 0.68552592,        nan,        nan,
              nan,        nan, 0.9015133 , 0.84389274, 0.81835176,
       0.87753779, 0.81818726, 0.76488295, 0.72891014,        nan,
              nan,        nan,        nan,        nan, 0.90244523,
       0.84434833, 0.84845122, 0.87864355, 0.82008685, 0.76692257,
              nan,        nan,        nan,        nan,        nan,
              nan, 0.90306946, 0.83735134, 0.85025966, 0.88054316,
       0.82257281,        nan,        nan,        nan,        nan,
              nan,        nan,        nan, 0.81679166, 0.83755